In [2]:
!pip install kagglehub

Defaulting to user installation because normal site-packages is not writeable


In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/home/studinstru/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pathlib import Path

print(Path.cwd())

for root, dirs, files in os.walk("."):
    if "train.csv" in files:
        print("FOUND:", root)

/home/studinstru/dlGenAI/dlgenai-t22026/notebooks


## Step 1: Setup & environment check

In [5]:
import sys, platform
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers

import matplotlib.pyplot as plt

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.system(), platform.release())
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("-" * 45)

# The device we'll use for all tensor computation.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device      :", device)
if device == "cuda":
    print("GPU         :", torch.cuda.get_device_name(0))
    print("VRAM (GB)   :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
print("-" * 45)

# Confirm the dataset files exist (paths are relative to this notebook's folder).
DATA_DIR = Path("../dataset/smart-mcq-solver-challenge")
for fname in ["train.csv", "test.csv", "sample_submission.csv"]:
    fpath = DATA_DIR / fname
    status = "FOUND   " if fpath.exists() else "MISSING!"
    print(f"{fname:22s} {status} -> {fpath}")

#import os
#from kaggle_secrets import UserSecretsClient
#os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

/home/studinstru/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


Python      : 3.10.12
Platform    : Linux 6.6.114.1-microsoft-standard-WSL2
NumPy       : 2.2.6
Pandas      : 2.3.3
PyTorch     : 2.5.1+cu121
Transformers: 5.13.0
---------------------------------------------
Device      : cuda
GPU         : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM (GB)   : 4.29
---------------------------------------------
train.csv              FOUND    -> ../dataset/smart-mcq-solver-challenge/train.csv
test.csv               FOUND    -> ../dataset/smart-mcq-solver-challenge/test.csv
sample_submission.csv  FOUND    -> ../dataset/smart-mcq-solver-challenge/sample_submission.csv


## Step 2: Reading inputs

In [6]:
train = pd.read_csv("../dataset/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("../dataset/smart-mcq-solver-challenge/test.csv")
sample_sub = pd.read_csv("../dataset/smart-mcq-solver-challenge/sample_submission.csv")

sample_sub.to_csv("submission.csv", index=False)

In [7]:
test.head()

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...


In [8]:
train.describe()

,id
count,2000.000000
mean,1000.500000
std,577.494589
min,1.000000
25%,500.750000
50%,1000.500000
75%,1500.250000
max,2000.000000


In [9]:
# check the shape of train and test
print(train.shape)
print(test.shape)
print(sample_sub.shape)

(2000, 8)
(500, 7)
(500, 2)


In [10]:
print("train columns:", list(train.columns))
print("test columns :", list(test.columns))
print()

print("Missing values per column (train):")
print(train.isna().sum().to_string())
print()

train columns: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']
test columns : ['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

Missing values per column (train):
id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0



## DLGenAI Project Milestone-4

This milestone focuses on formulating the Smart MCQ Solver Challenge as a proper multiple-choice classification problem. You will learn how to convert each prompt and its five options into model-ready inputs, use AutoModelForMultipleChoice to produce logits for A-E, apply LoRA for efficient fine-tuning, and run a small Hugging Face Trainer fine-tuning pipeline.

Suggested Readings: 
- Hugging Face Multiple Choice Task 
- LoRA Documentation 
- PyTorch Softmax

Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

Multiple-Choice Data Formatting
In this section, you will convert the Kaggle MCQ format into the structure required by multiple-choice models. Each question has one prompt and five options and each option must be paired with the prompt separately.

Q1. Label 

Convert the answer column in train.csv into numeric labels using the following mapping:
- A = 0
- B = 1
- C = 2
- D = 3
- E = 4

What is the encoded numeric label for the row at index 150?

In [12]:
# Mapping as specified
mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

# Create the numeric label column
train['label'] = train['answer'].map(mapping)

# Check the row at index 150
print(train.loc[150, ['id', 'answer', 'label']])

id        151
answer      C
label       2
Name: 150, dtype: object


Q2. Prompt-Option Formatting

For row index 0, create the Option B input using exactly this format:
str(prompt) + " [SEP] " + str(option_B)

What is the exact character length of this formatted input string?

In [14]:
row = train.loc[0]
format = str(row["prompt"]) + " [SEP] " + str(row["B"])
print(format)
print(len(format))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
407


Tokenization for Multiple-Choice Models
Multiple-choice models expect inputs in the shape:
batch_size x num_choices x sequence_length

Since each question has five options, every row becomes five tokenized sequences.

Q3. Single-Row MCQ Tokenization

Using bert-base-uncased, tokenize the five formatted inputs for row index 0 with:
- padding = "max_length"
- truncation = True
- max_length = 128
- return_tensors = "pt"

After reshaping for a multiple-choice model, the final input_ids tensor has shape:
[1, 5, 128]

What is the value of the second dimension?

In [17]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

row = train.loc[0]

options = ['A', 'B', 'C', 'D', 'E']
formatted_inputs = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]


encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# encoding['input_ids'] shape is [5, 128] here (5 separate sequences)
input_ids = encoding['input_ids'].unsqueeze(0)  # add batch dim -> [1, 5, 128]

print(input_ids.shape)



torch.Size([1, 5, 128])


Q4. Batch MCQ Tokenization

Tokenize the first 16 rows of train.csv as multiple-choice examples.
Each row has 5 choices.
Each choice is tokenized to length 128.

The final input_ids tensor has shape:
[16, 5, 128]

How many total token positions are in this tensor?

In [18]:
import pandas as pd
from transformers import AutoTokenizer

# Load data and take first 16 rows
subset = train.iloc[:16]

options = ['A', 'B', 'C', 'D', 'E']

# Build the flattened list of (prompt, option) formatted strings
# Order: question0-A, question0-B, ..., question0-E, question1-A, ...
formatted_inputs = []
for _, row in subset.iterrows():
    for opt in options:
        formatted_inputs.append(str(row['prompt']) + " [SEP] " + str(row[opt]))

# Tokenize all 16*5 = 80 sequences at once
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape from [80, 128] -> [16, 5, 128]
num_questions = len(subset)
num_choices = len(options)
input_ids = encoding['input_ids'].view(num_questions, num_choices, 128)

print("Shape:", input_ids.shape)          # torch.Size([16, 5, 128])
print("Total token positions:", input_ids.numel())  # 16 * 5 * 128

Shape: torch.Size([16, 5, 128])
Total token positions: 10240


Multiple-Choice Model Outputs

AutoModelForMultipleChoice produces one logit score for each answer option. For this competition, the model outputs five logits corresponding to A, B, C, D, and E.

Q5. Multiple-Choice Logits

Load bert-base-uncased using AutoModelForMultipleChoice.
Tokenize row index 0 as 5 choices and pass it through the model.

The output logits tensor has shape:
[1, 5]

How many logits are produced for one question

In [19]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# Load data and get row 0
row = train.loc[0]

options = ['A', 'B', 'C', 'D', 'E']
formatted_inputs = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

# Load tokenizer and multiple-choice model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

# Tokenize the 5 choices
encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [batch_size=1, num_choices=5, seq_len=128]
input_ids = encoding['input_ids'].unsqueeze(0)
attention_mask = encoding['attention_mask'].unsqueeze(0)
token_type_ids = encoding['token_type_ids'].unsqueeze(0)

# Forward pass (no gradients needed for inference)
model.eval()
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    )

logits = outputs.logits
print("Logits shape:", logits.shape)   # torch.Size([1, 5])
print("Logits:", logits)
print("Number of logits for this question:", logits.shape[1])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 402.17it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consi

Logits shape: torch.Size([1, 5])
Logits: tensor([[0.1339, 0.1430, 0.1523, 0.1377, 0.1631]])
Number of logits for this question: 5


Q6. Supervised Loss Tensor

For row index 0, pass the tokenized 5-choice input into AutoModelForMultipleChoice along with the correct encoded label.

The model returns a scalar loss tensor.

How many dimensions does this loss tensor have?

In [21]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# Load data and get row 0
row = train.loc[0]

options = ['A', 'B', 'C', 'D', 'E']
formatted_inputs = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

# Encode the correct answer label (A=0, B=1, C=2, D=3, E=4)
mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
label = torch.tensor([mapping[row['answer']]])  # shape [1] -> batch of 1

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

# Tokenize the 5 choices
encoding = tokenizer(
    formatted_inputs,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [batch_size=1, num_choices=5, seq_len=128]
input_ids = encoding['input_ids'].unsqueeze(0)
attention_mask = encoding['attention_mask'].unsqueeze(0)
token_type_ids = encoding['token_type_ids'].unsqueeze(0)

# Forward pass WITH labels -> model computes loss internally
outputs = model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    token_type_ids=token_type_ids,
    labels=label
)

loss = outputs.loss
print("Loss:", loss)
print("Loss shape:", loss.shape)
print("Number of dimensions:", loss.dim())

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3613.54it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Loss: tensor(1.6287, grad_fn=<NllLossBackward0>)
Loss shape: torch.Size([])
Number of dimensions: 0


LoRA for Efficient Fine-Tuning

LoRA freezes most of the original model and trains only a small number of adapter parameters. This makes fine-tuning faster and more memory-efficient.

Q7. LoRA Trainable Parameters

Apply LoRA to the bert-base-uncased multiple-choice model using:
r = 8
lora_alpha = 16
target_modules = ["query", "value"]
lora_dropout = 0.1
bias = "none"
task_type = TaskType.SEQ_CLS

Count trainable parameters using:
sum(p.numel() for p in model.parameters() if p.requires_grad)

How many parameters are trainable?

In [23]:
!pip install peft

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 KB 9.3 MB/s eta 0:00:0000:0100:01


In [24]:
from transformers import AutoModelForMultipleChoice
from peft import LoraConfig, get_peft_model, TaskType

# Load the base multiple-choice model
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

# Wrap the model with LoRA adapters
lora_model = get_peft_model(model, lora_config)

# Count trainable parameters
trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in lora_model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)
print(f"Trainable %: {100 * trainable_params / total_params:.4f}%")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1326.82it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Trainable params: 295681
Total params: 109778690
Trainable %: 0.2693%


Preparing Data for Hugging Face Trainer

Before training, the dataset must be converted into a format that the Hugging Face Trainer can understand: tokenized input_ids, attention_mask, and numeric labels.

Q8. Hugging Face Dataset Preparation

Create a Hugging Face Dataset from the first 100 rows of train.csv.

For each row, create:
input_ids with shape [5, 128]
attention_mask with shape [5, 128]
labels as the encoded answer label

For the first dataset item, input_ids has shape:
[5, 128]

How many tokenized choices are stored in input_ids?
*


In [26]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer

# Load first 100 rows
df = pd.read_csv('../dataset/smart-mcq-solver-challenge/train.csv').iloc[:100].reset_index(drop=True)

mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
options = ['A', 'B', 'C', 'D', 'E']

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def preprocess(row):
    # Build the 5 "prompt [SEP] option" pairs for this question
    formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

    encoding = tokenizer(
        formatted,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    return {
        "input_ids": encoding["input_ids"].tolist(),          # shape [5, 128]
        "attention_mask": encoding["attention_mask"].tolist(),# shape [5, 128]
        "labels": mapping[row["answer"]],
    }

# Convert pandas -> HF Dataset first, then map the preprocessing function over it
hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.map(preprocess)

# Check the first item
first_item = hf_dataset[0]
input_ids = torch.tensor(first_item["input_ids"])

print("input_ids shape:", input_ids.shape)
print("Number of tokenized choices:", input_ids.shape[0])
print("Label:", first_item["labels"])

Map: 100%|██████████| 100/100 [00:00<00:00, 338.35 examples/s]

input_ids shape: torch.Size([5, 128])
Number of tokenized choices: 5
Label: 1


Tiny Fine-Tuning and Inference

In this section, you will run a very small LoRA fine-tuning job using Hugging Face Trainer. Then you will use the fine-tuned model to produce probabilities for the answer options.

Q9. Tiny LoRA Fine-Tuning

Fine-tune a LoRA multiple-choice model on the first 32 rows using Hugging Face Trainer.

Use the following settings:
max_length = 64
per_device_train_batch_size = 4
gradient_accumulation_steps = 1
max_steps = 4

What is the final global_step reported by the Trainer?

In [29]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

# ---- Data: first 32 rows ----
df = pd.read_csv('../dataset/smart-mcq-solver-challenge/train.csv').iloc[:32].reset_index(drop=True)
mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
options = ['A', 'B', 'C', 'D', 'E']

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def preprocess(row):
    formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]
    encoding = tokenizer(
        formatted, 
        padding="max_length", 
        truncation=True,
        max_length=64,          # max_length = 64
        return_tensors="pt"          
    )
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": mapping[row["answer"]],
    }

hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.map(preprocess, remove_columns=hf_dataset.column_names)
hf_dataset.set_format(type="torch")

# ---- Model + LoRA ----
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(model, lora_config)

# ---- Training arguments ----
training_args = TrainingArguments(
    output_dir="./lora_mcq_tiny",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=hf_dataset,
)

train_result = trainer.train()

print("Final global_step:", trainer.state.global_step)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1734.08it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Step,Training Loss
1,1.658387
2,1.635692
3,1.612755
4,1.696022


Final global_step: 4


Q10. Probability Assigned to Option E After Fine-Tuning

Using the fine-tuned LoRA model from Q9, run inference on row index 0 and apply softmax to the logits.

What is the probability assigned to Option E?

Round your answer to 4 decimal places.

In [32]:
import torch
import torch.nn.functional as F
from transformers import set_seed

set_seed(42)  # do this BEFORE loading model/LoRA, for reproducibility

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

# ---- Data: first 32 rows ----
df = pd.read_csv('../dataset/smart-mcq-solver-challenge/train.csv').iloc[:32].reset_index(drop=True)
mapping = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
options = ['A', 'B', 'C', 'D', 'E']

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def preprocess(row):
    formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]
    encoding = tokenizer(
        formatted, 
        padding="max_length", 
        truncation=True,
        max_length=64,          # max_length = 64
        return_tensors="pt"          
    )
    return {
        "input_ids": encoding["input_ids"],
        "attention_mask": encoding["attention_mask"],
        "labels": mapping[row["answer"]],
    }

hf_dataset = Dataset.from_pandas(df)
hf_dataset = hf_dataset.map(preprocess, remove_columns=hf_dataset.column_names)
hf_dataset.set_format(type="torch")

# ---- Model + LoRA ----
model = AutoModelForMultipleChoice.from_pretrained('bert-base-uncased')

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)
lora_model = get_peft_model(model, lora_config)

# ---- Training arguments ----
training_args = TrainingArguments(
    output_dir="./lora_mcq_tiny",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=hf_dataset,
)

train_result = trainer.train()


import torch
import torch.nn.functional as F

lora_model.eval()

# Find out which device the model is actually on
device = next(lora_model.parameters()).device
print("Model is on:", device)

row = df.loc[0]
options = ['A', 'B', 'C', 'D', 'E']
formatted = [str(row['prompt']) + " [SEP] " + str(row[opt]) for opt in options]

encoding = tokenizer(
    formatted, padding="max_length", truncation=True,
    max_length=64, return_tensors="pt"
)

input_ids = encoding["input_ids"].unsqueeze(0).to(device)         # move to same device
attention_mask = encoding["attention_mask"].unsqueeze(0).to(device)

with torch.no_grad():
    outputs = lora_model(input_ids=input_ids, attention_mask=attention_mask)

logits = outputs.logits.to("cpu")   # bring back to CPU for easy printing/numpy conversion
probs = F.softmax(logits, dim=1)

print("Probabilities (A,B,C,D,E):", probs)
prob_E = probs[0, 4].item()
print(f"Probability assigned to Option E: {prob_E:.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1417.63it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Step,Training Loss
1,1.658387
2,1.635692
3,1.612755
4,1.696022


Model is on: cuda:0
Probabilities (A,B,C,D,E): tensor([[0.2031, 0.2005, 0.2028, 0.1966, 0.1970]])
Probability assigned to Option E: 0.1970
